In [1]:
import requests
import os

files_config = {
    'orders.csv': 'https://raw.githubusercontent.com/KirkYagami/DataEngineeringTrainingJune2025/refs/heads/main/04_BigData/05_Spark/03_Datasets/orders.csv',
    'products.csv': 'https://raw.githubusercontent.com/KirkYagami/DataEngineeringTrainingJune2025/refs/heads/main/04_BigData/05_Spark/03_Datasets/products.csv'
}

def download_file(url, filename):
    """Download a file from URL and save it locally."""
    try:
        print(f"Downloading {filename}...")
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(filename, 'wb') as file:
            file.write(response.content)

        print(f"✓ Successfully downloaded {filename} ({len(response.content)} bytes)")
        return True

    except requests.exceptions.RequestException as e:
        print(f"✗ Error downloading {filename}: {e}")
        return False
    except IOError as e:
        print(f"✗ Error saving {filename}: {e}")
        return False

def main():
    """Download all configured files."""
    print("Starting file downloads...")
    print(f"Current directory: {os.getcwd()}")
    print("-" * 50)

    success_count = 0
    total_files = len(files_config)

    for filename, url in files_config.items():
        if download_file(url, filename):
            success_count += 1
        print()

    print("-" * 50)
    print(f"Download complete: {success_count}/{total_files} files downloaded successfully")


    downloaded_files = [f for f in files_config.keys() if os.path.exists(f)]
    if downloaded_files:
        print("\nFiles in current directory:")
        for filename in downloaded_files:
            size = os.path.getsize(filename)
            print(f"  {filename} ({size:,} bytes)")

if __name__ == "__main__":
    main()

Starting file downloads...
Current directory: /content
--------------------------------------------------
✓ Successfully downloaded orders.csv (26248 bytes)

✓ Successfully downloaded products.csv (3999 bytes)

--------------------------------------------------
Download complete: 2/2 files downloaded successfully

Files in current directory:
  orders.csv (26,248 bytes)
  products.csv (3,999 bytes)


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("Learning Bucketing in PySpark") \
.getOrCreate()

In [4]:
df_orders = spark.read \
  .format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("orders.csv")

df_orders.show(5, False)
df_orders.printSchema()

+--------+----------+-----------+--------+-------------------+------------+
|order_id|product_id|customer_id|quantity|order_date         |total_amount|
+--------+----------+-----------+--------+-------------------+------------+
|1       |80        |10         |4       |2023-03-20 00:00:00|1003        |
|2       |69        |30         |3       |2023-12-11 00:00:00|780         |
|3       |61        |20         |4       |2023-04-26 00:00:00|1218        |
|4       |62        |44         |3       |2023-08-26 00:00:00|2022        |
|5       |78        |46         |4       |2023-08-05 00:00:00|1291        |
+--------+----------+-----------+--------+-------------------+------------+
only showing top 5 rows

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- total_amount: integer (nullable = true)



In [5]:
df_products = spark.read \
  .format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("products.csv")

df_products.show(5, False)
df_products.printSchema()

+----------+------------+-----------+-------+-----+-----+
|product_id|product_name|category   |brand  |price|stock|
+----------+------------+-----------+-------+-----+-----+
|1         |Product_1   |Electronics|Brand_4|26   |505  |
|2         |Product_2   |Apparel    |Brand_4|489  |15   |
|3         |Product_3   |Apparel    |Brand_4|102  |370  |
|4         |Product_4   |Groceries  |Brand_1|47   |433  |
|5         |Product_5   |Groceries  |Brand_3|244  |902  |
+----------+------------+-----------+-------+-----+-----+
only showing top 5 rows

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- stock: integer (nullable = true)



In [6]:
df_products.select("product_id").distinct().count()

100

In [7]:
df_orders.select("order_id").distinct().count()

1000

## Bucketing In Joins

In [8]:
df_orders_product_details = (
    df_orders.join(
        df_products,
        on="product_id",
        how="inner"
    )
)

In [9]:
df_orders_product_details.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#18, order_id#17, customer_id#19, quantity#20, order_date#21, total_amount#22, product_name#79, category#80, brand#81, price#82, stock#83]
   +- BroadcastHashJoin [product_id#18], [product_id#78], Inner, BuildRight, false
      :- Filter isnotnull(product_id#18)
      :  +- FileScan csv [order_id#17,product_id#18,customer_id#19,quantity#20,order_date#21,total_amount#22] Batched: false, DataFilters: [isnotnull(product_id#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,product_id:int,customer_id:int,quantity:int,order_date:timestamp,total_amount...
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=258]
         +- Filter isnotnull(product_id#78)
            +- FileScan csv [product_id#78,product_name#79,category#80,brand

In [10]:
df_orders_product_details.count()

1000

In [11]:
(
    df_products
    .write.bucketBy(4, col="product_id")
    .mode("overwrite")
    .saveAsTable("products_bucketed")
)

In [12]:
(
    df_orders
    .write.bucketBy(4, col="product_id")
    .mode("overwrite")
    .saveAsTable("orders_bucketed")
)

In [13]:
df_orders_bucketed = spark.table("orders_bucketed")
df_products_bucketed = spark.table("products_bucketed")

In [14]:
df_orders_product_details_bucketed = (
    df_orders_bucketed.join(
        df_products_bucketed,
        on="product_id",
        how="inner"
    )
)

In [15]:
df_orders_product_details_bucketed.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#192, order_id#191, customer_id#193, quantity#194, order_date#195, total_amount#196, product_name#204, category#205, brand#206, price#207, stock#208]
   +- BroadcastHashJoin [product_id#192], [product_id#203], Inner, BuildRight, false
      :- Filter isnotnull(product_id#192)
      :  +- FileScan parquet spark_catalog.default.orders_bucketed[order_id#191,product_id#192,customer_id#193,quantity#194,order_date#195,total_amount#196] Batched: true, Bucketed: false (disabled by query planner), DataFilters: [isnotnull(product_id#192)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/spark-warehouse/orders_bucketed], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,product_id:int,customer_id:int,quantity:int,order_date:timestamp,total_amount...
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [pl

In [19]:
df_orders_product_details_bucketed.count()

1000

## Bucketing In Aggregations

In [20]:
df_orders.show(5, False)

+--------+----------+-----------+--------+-------------------+------------+
|order_id|product_id|customer_id|quantity|order_date         |total_amount|
+--------+----------+-----------+--------+-------------------+------------+
|1       |80        |10         |4       |2023-03-20 00:00:00|1003        |
|2       |69        |30         |3       |2023-12-11 00:00:00|780         |
|3       |61        |20         |4       |2023-04-26 00:00:00|1218        |
|4       |62        |44         |3       |2023-08-26 00:00:00|2022        |
|5       |78        |46         |4       |2023-08-05 00:00:00|1291        |
+--------+----------+-----------+--------+-------------------+------------+
only showing top 5 rows



In [16]:
# WITHOUT BUCKETING

df_product_sales = (
    df_orders
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#18], functions=[sum(total_amount#22)])
   +- Exchange hashpartitioning(product_id#18, 200), ENSURE_REQUIREMENTS, [plan_id=480]
      +- HashAggregate(keys=[product_id#18], functions=[partial_sum(total_amount#22)])
         +- FileScan csv [product_id#18,total_amount#22] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<product_id:int,total_amount:int>




In [17]:
# WITH BUCKETING

df_product_sales = (
    df_orders_bucketed
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#192], functions=[sum(total_amount#196)])
   +- HashAggregate(keys=[product_id#192], functions=[partial_sum(total_amount#196)])
      +- FileScan parquet spark_catalog.default.orders_bucketed[product_id#192,total_amount#196] Batched: true, Bucketed: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/spark-warehouse/orders_bucketed], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<product_id:int,total_amount:int>, SelectedBucketsCount: 4 out of 4




## Bucket Pruning

In [18]:
df_product_sales_bucket_pruning = (
    df_orders_bucketed
    .filter(F.col("product_id") == 1)
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales_bucket_pruning.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#192], functions=[sum(total_amount#196)])
   +- HashAggregate(keys=[product_id#192], functions=[partial_sum(total_amount#196)])
      +- Filter (isnotnull(product_id#192) AND (product_id#192 = 1))
         +- FileScan parquet spark_catalog.default.orders_bucketed[product_id#192,total_amount#196] Batched: true, Bucketed: true, DataFilters: [isnotnull(product_id#192), (product_id#192 = 1)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/spark-warehouse/orders_bucketed], PartitionFilters: [], PushedFilters: [IsNotNull(product_id), EqualTo(product_id,1)], ReadSchema: struct<product_id:int,total_amount:int>, SelectedBucketsCount: 1 out of 4


